# Complejidad y actividad espectral del EEG a través de las etapas del sueño
### Pipeline reproducible de Ingeniería de Datos sobre Sleep-EDF Expanded (PhysioNet)

**Trabajo Práctico Final — Introducción a la Ingeniería de Datos**

Este notebook es el documento principal del trabajo. La lógica reutilizable vive en `src/` y se importa desde aquí;
el notebook ejecuta las etapas, muestra las validaciones y documenta cada decisión metodológica.

# Declaración de uso de IA

| Herramienta | Uso |
|---|---|
| OpenAI Codex | Diseño de la arquitectura del proyecto, generación y revisión de código Python y documentación técnica. |
| Claude Code — modelo Claude Opus 5.5 (`claude-opus-5-5`); versión de la herramienta: **TODO completar** | Diseño de la arquitectura del proyecto, generación y revisión de código Python y documentación técnica. |

Todas las decisiones metodológicas fueron revisadas y quedan documentadas en este notebook. Las herramientas de IA no
generaron datos: todos los resultados provienen de la ejecución del pipeline sobre datos públicos.

## Objetivo

Construir un pipeline de datos **reproducible de punta a punta** que:

1. descarga automáticamente EEG de sueño público (EDF/EDF+);
2. lo organiza según **BIDS**;
3. lo transforma (ETL) en epochs de 30 s etiquetados con su etapa de sueño;
4. calcula métricas espectrales y de complejidad por epoch y canal;
5. detecta y documenta datos faltantes, outliers estadísticos y artefactos de señal;
6. carga los resultados en una base **SQLite transaccional normalizada (3NF)**;
7. construye un **Data Warehouse en esquema estrella** y lo consulta con SQL;
8. comunica los resultados con visualizaciones Plotly y un dashboard Streamlit.

El objetivo principal **no** es demostrar una hipótesis neurocientífica nueva, sino demostrar un pipeline de
Ingeniería de Datos usando una pregunta neurocientífica real como caso de estudio.

## Pregunta de investigación

> **¿Cómo varían la complejidad y la actividad espectral del EEG entre las distintas etapas del sueño?**

- *Actividad espectral*: potencia absoluta y relativa en las bandas delta, theta, alfa, sigma y beta.
- *Complejidad*: Sample Entropy (impredecibilidad temporal) y Spectral Entropy (dispersión de la potencia en frecuencia).

Todos los resultados son **descriptivos**: no se realizan tests de hipótesis, por lo que una diferencia observada no
implica significancia estadística, y una asociación no implica causalidad.

## Dataset

**Sleep-EDF Expanded** (Kemp et al., 2000; PhysioNet, DOI [10.13026/C2X676](https://doi.org/10.13026/C2X676)),
subconjunto *sleep-cassette* (SC): polisomnografías ambulatorias de ~20 h de personas sanas sin medicación.

| Elemento | Descripción |
|---|---|
| `SC4ssN?0-PSG.edf` | Polisomnografía: EEG Fpz-Cz y Pz-Oz (100 Hz), EOG (100 Hz), EMG, respiración, temperatura (1 Hz) |
| `SC4ssN??-Hypnogram.edf` | Etapas de sueño puntuadas por expertos (reglas de Rechtschaffen & Kales) en segmentos múltiplos de 30 s |
| `SC-subjects.xls` | Edad, sexo y hora de apagado de luces por sujeto y noche |
| `SHA256SUMS.txt` | Lista oficial de archivos con sus hashes |

`ss` = sujeto, `N` = noche. El sufijo del hipnograma (p. ej. `EC`, `EH`) **difiere** del del PSG (`E0`), por lo que
el emparejamiento se hace por (sujeto, noche) y no por nombre de archivo.

Para desarrollo se usaron 2 sujetos; la versión final procesa **10 sujetos × 2 noches** cambiando un único parámetro
(`N_SUBJECTS` en `src/config.py`), sin modificar la lógica.

## Arquitectura general

```
PhysioNet (AWS Open Data mirror / physionet.org)
    │  extract.py      descarga + SHA256 + manifest
    ▼
data/raw/            EDF originales (no se modifican)
    │  bids.py         MNE-BIDS
    ▼
data/bids/           sub-XX/ses-N/eeg/  + participants.tsv, events.tsv, channels.tsv
    │  preprocessing.py  lectura, validación, epochs, recorte, filtro
    │  features.py       Welch, potencias, entropías, RMS, pico a pico
    │  quality.py        artefactos, outliers, faltantes
    │  pipeline.py       orquestación + caché por registro
    ▼
data/processed/      features_tidy.parquet (epoch × canal), epochs, recordings, reportes
    │  database_oltp.py
    ▼
data/database/sleep_oltp.sqlite   modelo relacional 3NF
    │  database_dw.py  ETL
    ▼
data/database/sleep_dw.sqlite     esquema estrella
    │  queries.py / plots.py
    ▼
Notebook (EDA)  +  Streamlit (src/app.py)
```

| Módulo | Responsabilidad |
|---|---|
| `config.py` | Rutas relativas, selección de sujetos, parámetros metodológicos y umbrales |
| `extract.py` | Catálogo, selección, descarga validada y manifest |
| `bids.py` | Conversión a BIDS y metadatos |
| `preprocessing.py` | Lectura EDF, validación, epoching, recorte de vigilia, filtrado |
| `features.py` | Métricas espectrales y de complejidad |
| `quality.py` | Artefactos, outliers estadísticos, reporte de faltantes |
| `pipeline.py` | Orquestación del Transform con caché por registro |
| `database_oltp.py` / `database_dw.py` | Bases SQLite transaccional y analítica |
| `queries.py` / `plots.py` | Consultas SQL y figuras (compartidas con el dashboard) |
| `validation.py` | Checks automáticos con mensajes comprensibles |
| `app.py` | Dashboard Streamlit |

## Configuración

La primera celda prepara el entorno. En **Google Colab** clona el repositorio e instala las dependencias; localmente
solo agrega la raíz del proyecto al `sys.path`. Todas las rutas se calculan relativas a la raíz con `pathlib`.

`QUICK_RUN = True` procesa 2 sujetos (≈5 min); `False` usa la configuración completa (10 sujetos × 2 noches,
≈20–30 min la primera vez; las ejecuciones siguientes reutilizan el caché).

In [1]:
import os, sys, subprocess
from pathlib import Path

QUICK_RUN = False  # True: 2 sujetos para una prueba rápida

# TODO: reemplazar por la URL del repositorio en GitHub antes de ejecutar en Colab.
REPO_URL = "https://github.com/<usuario>/<repositorio>.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_dir = Path("/content") / Path(REPO_URL).stem
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    PROJECT_ROOT = repo_dir
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Raíz del proyecto:", PROJECT_ROOT)

Raíz del proyecto: C:\workspace\IID_TPF


In [2]:
import json, logging, random, warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

from src import (config, extract, bids, preprocessing, features, quality, pipeline,
                 database_oltp, database_dw, queries, plots, validation)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
logging.getLogger("mne").setLevel(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

random.seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)

N_SUBJECTS = 2 if QUICK_RUN else config.N_SUBJECTS

def show_checks(lines):
    """Muestra el resultado de un bloque de validaciones (cada una lanza ValidationError si falla)."""
    for line in lines:
        print("  ✔", line)

print("Sujetos a procesar:", N_SUBJECTS, "| noches:", config.NIGHTS)
print("Canales EEG:", config.EEG_CHANNELS)
print("Epoch:", config.EPOCH_DURATION_S, "s | filtro:", config.L_FREQ_HZ, "-", config.H_FREQ_HZ, "Hz",
      "| vigilia conservada:", config.WAKE_PADDING_MIN, "min")
print("Bandas:", config.FREQ_BANDS)
print("Umbrales de artefacto: pico a pico >", config.ARTIFACT_MAX_PTP_UV, "µV; SD <", config.ARTIFACT_MIN_STD_UV, "µV")
print("Versiones:", pipeline.library_versions())

Sujetos a procesar: 10 | noches: (1, 2)
Canales EEG: ['EEG Fpz-Cz', 'EEG Pz-Oz']
Epoch: 30.0 s | filtro: 0.5 - 30.0 Hz | vigilia conservada: 30 min
Bandas: {'delta': (0.5, 4.0), 'theta': (4.0, 8.0), 'alpha': (8.0, 12.0), 'sigma': (12.0, 16.0), 'beta': (16.0, 30.0)}
Umbrales de artefacto: pico a pico > 500.0 µV; SD < 0.5 µV
Versiones: {'python': '3.13.11', 'numpy': '2.2.6', 'pandas': '2.2.3', 'scipy': '1.15.3', 'mne': '1.9.0', 'mne-bids': '0.20.0', 'antropy': '0.1.9', 'pyarrow': '25.0.1', 'plotly': '7.1.0', 'streamlit': '1.65.0'}


## Extract

```
PhysioNet
    ↓
EDF PSG + Hypnogram
    ↓
data/raw/
```

1. Se descarga `SHA256SUMS.txt`, la lista oficial de archivos del dataset con sus hashes.
2. Con ella se arma el **catálogo** de registros SC, emparejando PSG e hipnograma por (sujeto, noche).
3. Se seleccionan los primeros `N_SUBJECTS` sujetos y las noches de `config.NIGHTS`.
4. Cada archivo se descarga **solo si no existe o no es válido**; la validación exige tamaño mínimo y que el SHA256
   coincida con el oficial. La descarga es atómica (`.part` + renombrado), por lo que una interrupción no deja
   archivos corruptos.
5. Se registra todo en `data/raw/download_manifest.csv` (archivo, tamaño, hash, estado, URL de origen).

**Fuente**: se usa primero el *mirror* oficial de PhysioNet en AWS Open Data (~50× más rápido en las pruebas) y, si
falla, `physionet.org`. Como cada archivo se verifica contra los hashes oficiales, la fuente no altera los datos.

In [3]:
catalog = extract.build_catalog(extract.fetch_checksums())
print(f"Catálogo sleep-cassette: {len(catalog)} registros de {catalog['subject'].nunique()} sujetos")
print("Sujetos con una sola noche:", sorted(catalog.groupby("subject")["night"].count().loc[lambda s: s < 2].index))

manifest = extract.run_extract(n_subjects=N_SUBJECTS)
display(manifest[["recording_id", "kind", "file", "size_bytes", "status"]])
show_checks(validation.validate_downloads(manifest))

INFO src.extract: Registros seleccionados: 20 (10 sujetos)


Catálogo sleep-cassette: 153 registros de 78 sujetos
Sujetos con una sola noche: [13, 36, 52]


INFO src.extract: PSG       SC4001E0-PSG.edf         cached


INFO src.extract: Hypnogram SC4001EC-Hypnogram.edf   cached


INFO src.extract: PSG       SC4002E0-PSG.edf         cached


INFO src.extract: Hypnogram SC4002EC-Hypnogram.edf   cached


... (34 outputs left out)

INFO src.extract: PSG       SC4092E0-PSG.edf         cached


INFO src.extract: Hypnogram SC4092EC-Hypnogram.edf   cached


,recording_id,kind,file,size_bytes,status
0,SC4001,PSG,data/raw/sleep-cassette/SC4001E0-PSG.edf,48338048,cached
1,SC4001,Hypnogram,data/raw/sleep-cassette/SC4001EC-Hypnogram.edf,4620,cached
2,SC4002,PSG,data/raw/sleep-cassette/SC4002E0-PSG.edf,51621248,cached
3,SC4002,Hypnogram,data/raw/sleep-cassette/SC4002EC-Hypnogram.edf,4580,cached
4,SC4011,PSG,data/raw/sleep-cassette/SC4011E0-PSG.edf,51110528,cached
5,SC4011,Hypnogram,data/raw/sleep-cassette/SC4011EH-Hypnogram.edf,3896,cached
6,SC4012,PSG,data/raw/sleep-cassette/SC4012E0-PSG.edf,51986048,cached
7,SC4012,Hypnogram,data/raw/sleep-cassette/SC4012EC-Hypnogram.edf,5082,cached
8,SC4021,PSG,data/raw/sleep-cassette/SC4021E0-PSG.edf,51147008,cached
9,SC4021,Hypnogram,data/raw/sleep-cassette/SC4021EH-Hypnogram.edf,4804,cached


  ✔ 40 archivos presentes en data/raw
  ✔ 20 registros con PSG + Hypnogram


## Organización BIDS

### EDF y BIDS no son lo mismo

- **EDF/EDF+** (*European Data Format*) es un **formato de archivo**: define cómo se codifican en binario las señales
  de un registro (header con canales, unidades, rango físico/digital, frecuencia de muestreo, y luego los bloques de
  datos). EDF+ agrega anotaciones, que en Sleep-EDF contienen el hipnograma.
- **BIDS** (*Brain Imaging Data Structure*) es una **convención de organización** de un *dataset* completo: nombres de
  carpetas y archivos (`sub-XX/ses-N/eeg/...`), y metadatos legibles en TSV/JSON (`participants.tsv`,
  `channels.tsv`, `events.tsv`, `dataset_description.json`). BIDS no reemplaza a EDF: **lo contiene** (EEG-BIDS
  admite EDF como formato de datos).

### Por qué BIDS mejora la reproducibilidad y la interoperabilidad

- Cualquier herramienta compatible (MNE-BIDS, EEGLAB, validadores BIDS) sabe dónde está cada dato y qué significa,
  sin código ad hoc para los nombres de Sleep-EDF.
- Los metadatos (sujetos, canales, eventos) quedan en texto plano, versionables y legibles por humanos.
- Separa claramente datos crudos, derivados y descripción del dataset.

### Decisiones

- `data/raw/` **no se modifica**. MNE-BIDS copia el EDF a `data/bids/` sin reescribir la señal (se verifica byte a
  byte durante el desarrollo).
- La sesión BIDS es la noche (`ses-1`, `ses-2`) y la tarea es `sleep`.
- Las etapas de sueño se exportan como eventos en `*_events.tsv`.
- `participants.tsv` se completa con edad y sexo de `SC-subjects.xls`, porque el EDF de Sleep-EDF no los incluye.
- `channels.tsv` se corrige con los valores del header EDF. MNE remuestrea internamente todos los canales a 100 Hz,
  pero el archivo real tiene EMG, respiración y temperatura a 1 Hz; los metadatos deben describir el archivo, no la
  representación en memoria.

In [4]:
bids_index = bids.run_bids(manifest)
display(bids_index)

print("Estructura (primer sujeto):")
first = sorted(config.BIDS_DIR.glob("sub-*"))[0]
for path in sorted(first.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(config.BIDS_DIR).as_posix())

display(pd.read_csv(config.BIDS_DIR / "participants.tsv", sep="\t"))
channels_tsv = next(first.rglob("*_channels.tsv"))
display(pd.read_csv(channels_tsv, sep="\t")[["name", "type", "units", "low_cutoff", "high_cutoff", "sampling_frequency"]])

INFO src.bids: BIDS existente, se reutiliza: sub-00_ses-1_task-sleep_eeg.edf


INFO src.bids: BIDS existente, se reutiliza: sub-00_ses-2_task-sleep_eeg.edf


INFO src.bids: BIDS existente, se reutiliza: sub-01_ses-1_task-sleep_eeg.edf


INFO src.bids: BIDS existente, se reutiliza: sub-01_ses-2_task-sleep_eeg.edf


INFO src.bids: BIDS existente, se reutiliza: sub-02_ses-1_task-sleep_eeg.edf


INFO src.bids: BIDS existente, se reutiliza: sub-02_ses-2_task-sleep_eeg.edf


... (14 outputs left out)

,recording_id,subject,night,bids_file
0,SC4001,0,1,data/bids/sub-00/ses-1/eeg/sub-00_ses-1_task-s...
1,SC4002,0,2,data/bids/sub-00/ses-2/eeg/sub-00_ses-2_task-s...
2,SC4011,1,1,data/bids/sub-01/ses-1/eeg/sub-01_ses-1_task-s...
3,SC4012,1,2,data/bids/sub-01/ses-2/eeg/sub-01_ses-2_task-s...
4,SC4021,2,1,data/bids/sub-02/ses-1/eeg/sub-02_ses-1_task-s...
5,SC4022,2,2,data/bids/sub-02/ses-2/eeg/sub-02_ses-2_task-s...
6,SC4031,3,1,data/bids/sub-03/ses-1/eeg/sub-03_ses-1_task-s...
7,SC4032,3,2,data/bids/sub-03/ses-2/eeg/sub-03_ses-2_task-s...
8,SC4041,4,1,data/bids/sub-04/ses-1/eeg/sub-04_ses-1_task-s...
9,SC4042,4,2,data/bids/sub-04/ses-2/eeg/sub-04_ses-2_task-s...


Estructura (primer sujeto):
   sub-00/ses-1/eeg/sub-00_ses-1_task-sleep_channels.tsv
   sub-00/ses-1/eeg/sub-00_ses-1_task-sleep_eeg.edf
   sub-00/ses-1/eeg/sub-00_ses-1_task-sleep_eeg.json
   sub-00/ses-1/eeg/sub-00_ses-1_task-sleep_events.json
   sub-00/ses-1/eeg/sub-00_ses-1_task-sleep_events.tsv
   sub-00/ses-1/sub-00_ses-1_scans.tsv
   sub-00/ses-2/eeg/sub-00_ses-2_task-sleep_channels.tsv
   sub-00/ses-2/eeg/sub-00_ses-2_task-sleep_eeg.edf
   sub-00/ses-2/eeg/sub-00_ses-2_task-sleep_eeg.json
   sub-00/ses-2/eeg/sub-00_ses-2_task-sleep_events.json
   sub-00/ses-2/eeg/sub-00_ses-2_task-sleep_events.tsv
   sub-00/ses-2/sub-00_ses-2_scans.tsv


,participant_id,age,sex
0,sub-00,33,F
1,sub-01,33,F
2,sub-02,26,F
3,sub-03,26,F
4,sub-04,34,F
5,sub-05,28,F
6,sub-06,31,F
7,sub-07,30,F
8,sub-08,25,F
9,sub-09,25,F


,name,type,units,low_cutoff,high_cutoff,sampling_frequency
0,EEG Fpz-Cz,EEG,µV,0.50,100.0,100.0
1,EEG Pz-Oz,EEG,µV,0.50,100.0,100.0
2,EOG horizontal,EOG,µV,0.50,100.0,100.0
3,Resp oro-nasal,RESP,NaN,0.03,0.9,1.0
4,EMG submental,EMG,µV,16.00,0.7,1.0
5,Temp rectal,TEMP,oC,NaN,NaN,1.0
6,Event marker,MISC,NaN,NaN,NaN,1.0


## Exploración inicial

Antes de procesar se inspecciona **cada** registro sin suponer valores: canales, frecuencia de muestreo y unidades
(leídas del header EDF), duración, anotaciones y coherencia temporal entre PSG e hipnograma. `validate_recording`
detiene el pipeline con un mensaje claro si falta un canal, la frecuencia o la unidad no son las esperadas, el PSG y
el hipnograma no comparten inicio, o aparece una etiqueta de etapa desconocida.

In [5]:
files = manifest.pivot(index=["recording_id", "subject", "night"], columns="kind", values="file").reset_index()
rows, warnings_found = [], []
for rec in files.itertuples(index=False):
    info = preprocessing.inspect_recording(config.PROJECT_ROOT / rec.PSG, config.PROJECT_ROOT / rec.Hypnogram)
    warnings_found += preprocessing.validate_recording(info)
    rows.append({
        "recording_id": rec.recording_id,
        "inicio": info["start_datetime"],
        "duración PSG [h]": round(info["psg_duration_s"] / 3600, 2),
        "fin hipnograma [h]": round(info["hypnogram_end_s"] / 3600, 2),
        "fs EEG [Hz]": sorted(set(info["eeg_sfreq_hz"].values())),
        "unidades EEG": sorted(set(info["eeg_units"].values())),
        "etiquetas": ", ".join(f"{k.replace('Sleep stage ', '')}:{v}" for k, v in sorted(info["annotation_counts"].items())),
    })
display(pd.DataFrame(rows))
print("Canales del PSG:", info["channels"])
print("Prefiltro de adquisición EEG:", info["eeg_prefilter"])
print(f"Advertencias ({len(warnings_found)}):")
for w in warnings_found:
    print("  -", w)

,recording_id,inicio,duración PSG [h],fin hipnograma [h],fs EEG [Hz],unidades EEG,etiquetas
0,SC4001,1989-04-24 16:13:00,22.08,24.0,[100.0],[uV],"1:24, 2:40, 3:48, 4:23, ?:1, R:6, W:12"
1,SC4002,1989-04-25 14:50:00,23.58,24.0,[100.0],[uV],"Movement time:1, 1:32, 2:37, 3:30, 4:16, ?:1, ..."
2,SC4011,1989-03-29 16:49:00,23.35,24.0,[100.0],[uV],"1:33, 2:40, 3:21, 4:8, ?:1, R:8, W:15"
3,SC4012,1989-03-30 17:00:00,23.75,24.0,[100.0],[uV],"1:42, 2:55, 3:29, 4:9, ?:1, R:18, W:17"
4,SC4021,1989-04-05 16:48:00,23.37,24.0,[100.0],[uV],"1:22, 2:56, 3:43, 4:15, ?:1, R:15, W:9"
5,SC4022,1989-04-06 16:42:00,22.97,24.0,[100.0],[uV],"Movement time:1, 1:52, 2:63, 3:34, 4:11, ?:1, ..."
6,SC4031,1989-04-10 16:50:00,23.50,24.0,[100.0],[uV],"1:30, 2:42, 3:21, 4:1, ?:1, R:10, W:14"
7,SC4032,1989-04-11 16:49:00,22.77,24.0,[100.0],[uV],"1:20, 2:33, 3:28, 4:16, ?:1, R:12, W:13"
8,SC4041,1989-04-24 16:50:00,21.42,24.0,[100.0],[uV],"Movement time:1, 1:43, 2:47, 3:15, ?:1, R:24, ..."
9,SC4042,1989-04-25 15:04:00,23.27,24.0,[100.0],[uV],"Movement time:4, 1:51, 2:48, 3:22, 4:3, ?:1, R..."


Canales del PSG: ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'Resp oro-nasal', 'EMG submental', 'Temp rectal', 'Event marker']
Prefiltro de adquisición EEG: {'EEG Fpz-Cz': 'HP:0.5Hz LP:100Hz [enhanced cassette BW]', 'EEG Pz-Oz': 'HP:0.5Hz LP:100Hz [enhanced cassette BW]'}
Advertencias (20):
  - SC4001E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 79500 s; los epochs sin señal se descartan al segmentar
  - SC4002E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 84900 s; los epochs sin señal se descartan al segmentar
  - SC4011E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 84060 s; los epochs sin señal se descartan al segmentar
  - SC4012E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 85500 s; los epochs sin señal se descartan al segmentar
  - SC4021E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 84120 s; los epochs sin señal se descartan al segmentar
  - SC4022E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 82680 s; los

Observaciones:

- Todos los registros tienen los dos canales EEG a 100 Hz, en µV, con prefiltro de hardware 0.5–100 Hz.
- Los conteos de etiquetas son **segmentos** del hipnograma, no epochs (un segmento W puede durar horas).
- En todos los registros el hipnograma se extiende más allá del final de la señal: esos epochs se descartan y se
  contabilizan como faltantes (ver *Tratamiento de datos faltantes*).

## Transform

Cada registro se procesa **secuencialmente**, para no tener varias señales en memoria:

1. **Epochs**: los segmentos del hipnograma se dividen en epochs de 30 s. Se verifica que las anotaciones sean
   contiguas y estén alineadas a 30 s; si hubiera huecos, el pipeline se detiene.
2. **Recorte de vigilia**: los registros SC duran ~20 h y la mayor parte es vigilia diurna. Se conservan solo
   **30 min de W antes del primer epoch de sueño y 30 min después del último** (práctica habitual con este dataset).
   Sin recorte, W representaría ~70 % de los epochs y dominaría cualquier comparación. Los epochs recortados no se
   eliminan de la tabla de epochs: quedan marcados (`in_analysis_window = False`).
3. **Carga selectiva**: solo se leen los 2 canales EEG. Esto evita que MNE suba a 100 Hz los canales de 1 Hz y reduce
   la memoria.
4. **Filtro pasa-banda 0.5–30 Hz** (FIR de fase cero, el valor por defecto de MNE), aplicado a la señal **continua**
   recortada, con 30 s de margen a cada lado, antes de segmentar. Filtrar cada epoch por separado introduciría
   efectos de borde.
   - El pasa-altos de 0.5 Hz elimina la deriva lenta. El hardware ya aplica 0.5 Hz, pero así se garantiza el mismo
     corte en todos los registros.
   - El pasa-bajos de 30 Hz coincide con el límite superior de la banda beta analizada y atenúa actividad muscular y
     ruido de red (50 Hz, cerca de Nyquist).
   - No se aplica ninguna otra transformación (ni re-referencia, ni ICA, ni normalización): para las métricas
     elegidas no son necesarias y agregarían supuestos.
5. **Array de epochs** en µV, float32 (precisión suficiente para la señal y la mitad de memoria). Las features se
   calculan en float64, porque acumular sumas espectrales en float32 introduce errores de ~1e-6.
6. **Features y flags de artefacto**, guardados como Parquet por registro (caché).

El caché se invalida automáticamente si cambian los parámetros metodológicos de `config.py` o la versión del código
del Transform (se guarda un hash de ambos).

A continuación se ilustra el efecto del filtro sobre un epoch y luego se ejecuta el pipeline completo.

In [6]:
import plotly.graph_objects as go

rec = files.iloc[0]
psg, hyp = config.PROJECT_ROOT / rec.PSG, config.PROJECT_ROOT / rec.Hypnogram
info = preprocessing.inspect_recording(psg, hyp)
ep = preprocessing.mark_analysis_window(
    preprocessing.annotations_to_epochs(preprocessing.read_hypnogram(hyp), info["psg_duration_s"]))
example = ep[ep["in_analysis_window"] & ep["has_signal"] & (ep["stage"] == "N2")].iloc[20]

raw_unfiltered = preprocessing.load_eeg(psg, hyp)
raw_filtered = preprocessing.crop_and_filter(raw_unfiltered.copy(), ep)
x_raw = preprocessing.extract_epoch_array(raw_unfiltered, np.array([example.start_sec]))[0, 0]
x_filt = preprocessing.extract_epoch_array(raw_filtered, np.array([example.start_sec]))[0, 0]
del raw_unfiltered, raw_filtered

t = np.arange(x_raw.size) / config.EXPECTED_SFREQ_HZ
fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=x_raw, name="Original", line=dict(color=plots.INK_MUTED, width=1)))
fig.add_trace(go.Scatter(x=t, y=x_filt, name="Filtrado 0.5–30 Hz", line=dict(color=plots.STAGE_COLORS["N2"], width=1)))
fig.update_xaxes(title="Tiempo dentro del epoch [s]")
fig.update_yaxes(title="Amplitud [µV]")
plots._style(fig, f"Fpz-Cz · {rec.recording_id} · epoch {example.epoch_number} (N2)", height=360).show()

In [7]:
results = pipeline.run_pipeline(manifest)
tidy, epochs, recordings, missing = results["tidy"], results["epochs"], results["recordings"], results["missing"]
print(f"Tabla tidy: {tidy.shape[0]:,} filas × {tidy.shape[1]} columnas, "
      f"{tidy.memory_usage(deep=True).sum() / 1e6:.1f} MB en memoria")

INFO src.pipeline: SC4001: caché reutilizado


INFO src.pipeline: SC4002: caché reutilizado


INFO src.pipeline: SC4011: caché reutilizado


INFO src.pipeline: SC4012: caché reutilizado


INFO src.pipeline: SC4021: caché reutilizado


INFO src.pipeline: SC4022: caché reutilizado


... (12 outputs left out)

INFO src.pipeline: SC4091: caché reutilizado


INFO src.pipeline: SC4092: caché reutilizado


INFO src.pipeline: Tidy: 42142 filas x 27 columnas (13.7 MB en memoria)


Tabla tidy: 42,142 filas × 27 columnas, 13.7 MB en memoria


## Hipnograma y epochs

Las etiquetas originales (R&K) se mapean a la nomenclatura AASM:

| Hipnograma | Etapa | Tratamiento |
|---|---|---|
| `Sleep stage W` | W | Puntuada |
| `Sleep stage 1` | N1 | Puntuada |
| `Sleep stage 2` | N2 | Puntuada |
| `Sleep stage 3`, `Sleep stage 4` | N3 | Puntuada (R&K S3 + S4 = AASM N3) |
| `Sleep stage R` | REM | Puntuada |
| `Movement time` | MOVE | **No es una etapa**: se conserva, no se imputa |
| `Sleep stage ?` | UNK | **Desconocida**: se conserva, no se imputa |

MOVE y `?` no se reasignan a una etapa vecina: hacerlo inventaría información que el experto no registró. Se
conservan con su propia categoría, figuran en las bases y se excluyen de las comparaciones entre etapas.

In [8]:
display(recordings[["recording_id", "subject", "night", "age", "sex", "lights_off",
                    "n_epochs_hypnogram", "n_epochs_analyzed"]])
counts = (epochs.assign(grupo=np.select(
              [~epochs["has_signal"], ~epochs["in_analysis_window"]],
              ["sin señal", "recortado (vigilia)"], "analizado"))
          .pivot_table(index="stage", columns="grupo", values="epoch_number", aggfunc="count", fill_value=0))
display(counts)

,recording_id,subject,night,age,sex,lights_off,n_epochs_hypnogram,n_epochs_analyzed
0,SC4001,0,1,33,F,00:38:00,2880,841
1,SC4002,0,2,33,F,21:57:00,2880,1128
2,SC4011,1,1,33,F,22:44:00,2880,1103
3,SC4012,1,2,33,F,22:15:00,2880,1186
4,SC4021,2,1,26,F,22:50:00,2880,1025
5,SC4022,2,2,26,F,22:57:00,2880,1010
6,SC4031,3,1,26,F,00:02:00,2880,952
7,SC4032,3,2,26,F,00:24:00,2880,911
8,SC4041,4,1,34,F,23:12:00,2880,1236
9,SC4042,4,2,34,F,23:35:00,2880,1204


grupo,analizado,recortado (vigilia),sin señal
stage,,,
MOVE,32,0,0
N1,1689,0,0
N2,8927,0,0
N3,3074,0,0
REM,3693,0,0
UNK,0,804,2266
W,3656,33459,0


In [9]:
obs_preview = tidy.rename(columns={"sleep_stage": "stage_code", "start_time_sec": "start_sec"})
obs_preview["channel"] = obs_preview["channel"].astype(str).str.replace("EEG ", "")
onset = (obs_preview[obs_preview["stage_code"].isin(["N1", "N2", "N3", "REM"])]
         .groupby("recording_id")["start_sec"].min())
obs_preview["hours_since_sleep_onset"] = (obs_preview["start_sec"] - obs_preview["recording_id"].map(onset)) / 3600
obs_preview["stage_code"] = obs_preview["stage_code"].astype(str)
plots.hypnogram(obs_preview, files.iloc[0].recording_id).show()

## Control de calidad

Además de las validaciones de entrada (sección *Exploración inicial*), sobre la tabla resultante se verifica que:

- no haya claves (registro, epoch, canal) duplicadas;
- cada epoch tenga todos los canales;
- todos los epochs empiecen en múltiplos de 30 s;
- las etapas sean válidas;
- la potencia relativa esté en [0, 1] y sume 1;
- `is_artifact` sea coherente con `artifact_reason`;
- no haya features NaN en filas sin artefacto.

In [10]:
show_checks(validation.validate_tidy(tidy))

  ✔ Sin claves (recording, epoch, channel) duplicadas
  ✔ Canales: ['EEG Fpz-Cz', 'EEG Pz-Oz']
  ✔ Cada epoch tiene todos los canales
  ✔ Inicios de epoch múltiplos de 30 s
  ✔ Etapas válidas: ['MOVE', 'N1', 'N2', 'N3', 'REM', 'W']
  ✔ Potencia relativa en [0, 1] y suma 1 ± 1e-09 (sin artefactos)
  ✔ is_artifact coherente con artifact_reason
  ✔ Sin features NaN en filas no artefactadas


## Tratamiento de datos faltantes

Se distinguen cuatro situaciones. Ninguna se resuelve por imputación:

1. **Epochs del hipnograma sin señal**: el hipnograma continúa después del final del PSG. Sin señal no hay
   features: se excluyen y se cuentan.
2. **Vigilia fuera de la ventana de análisis**: no es un faltante, sino una **exclusión por diseño** (recorte de
   vigilia). Se reporta para que el volumen de datos descartado sea explícito.
3. **Etapa desconocida (`?`) o Movement time dentro de la ventana**: la señal existe y se calculan sus features, pero
   la etapa es desconocida. Se conservan como UNK/MOVE y se excluyen de las comparaciones entre etapas.
   Transformarlas en una etapa (por ejemplo, copiando la del epoch anterior) sería fabricar una etiqueta.
4. **Features no calculables**: si la señal es plana o tiene valores no finitos, la Sample Entropy, la Spectral
   Entropy y las potencias relativas no están definidas. Se guardan como NaN (`NULL` en SQL) y no como 0, porque
   `antropy` devolvería 0, que se leería como "regularidad perfecta".

In [11]:
display(missing.style.format({"percent": "{:.2f} %"}).hide(axis="index"))

level,item,count,denominator,percent,cause,decision
epoch,Epoch del hipnograma sin señal EEG,2266,57600,3.93 %,El hipnograma se extiende más allá del final del PSG,Excluido: no hay señal de la cual extraer features
epoch,Vigilia fuera de la ventana de análisis,34263,57600,59.48 %,Registros de ~20 h; se conservan 30 min de W antes/después del sueño,Excluido por diseño (no es un faltante): evita que W domine el análisis
epoch,Etapa desconocida ('?') en ventana,0,57600,0.00 %,El experto no asignó etapa,Se conserva con etapa UNK; no se imputa; se excluye de comparaciones entre etapas
epoch,Movement time en ventana,32,57600,0.06 %,Movimiento corporal: el experto no pudo asignar etapa,Se conserva con etapa MOVE; no se imputa; se excluye de comparaciones entre etapas
epoch,Etapa sin mapeo,0,57600,0.00 %,Descripción del hipnograma no contemplada en STAGE_MAPPING,Debe ser 0 (la validación de cada registro lo impide)
feature,delta_abs = NaN,0,42142,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto
feature,theta_abs = NaN,0,42142,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto
feature,alpha_abs = NaN,0,42142,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto
feature,sigma_abs = NaN,0,42142,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto
feature,beta_abs = NaN,0,42142,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto


## Tratamiento de outliers y artefactos

### Outlier estadístico ≠ artefacto de señal

| | Outlier estadístico | Artefacto de señal |
|---|---|---|
| Definición | Valor extremo respecto de la distribución observada | El registro no refleja actividad cerebral |
| Criterio | Regla de Tukey: fuera de [Q1 − 1.5·IQR, Q3 + 1.5·IQR] | Criterios físicos sobre la señal |
| Ejemplo | Una onda lenta de gran amplitud en N3 | Electrodo desconectado, saturación, movimiento |
| Decisión | **Solo se marca** (`is_stat_outlier`, `outlier_features`) | **Se marca** (`is_artifact`, `artifact_reason`) y se excluye por defecto de los análisis |

**Ninguna fila se elimina.** Un valor extremo puede ser fisiología real, y eliminarlo sesgaría justamente las etapas
más interesantes.

### Criterios de artefacto (configurables en `config.py`)

- `non_finite`: algún valor NaN/inf en el epoch.
- `flat_signal`: desvío estándar < `ARTIFACT_MIN_STD_UV` (0.5 µV), compatible con un electrodo desconectado.
- `excessive_amplitude`: pico a pico > `ARTIFACT_MAX_PTP_UV` (500 µV), un valor no fisiológico para EEG de
  superficie. La amplitud fisiológica más alta corresponde a las ondas lentas de N3 en Fpz-Cz: percentil 99 ≈ 350 µV
  y máximo observado ≈ 435 µV. Un umbral más bajo (por ejemplo 300 µV) marcaría sueño profundo como artefacto.

### Outliers estadísticos

La regla de Tukey se aplica **por canal × etapa**, porque las distribuciones difieren legítimamente entre etapas (N3
tiene más delta que REM por definición). Las potencias absolutas se evalúan en log10, porque su distribución es muy
asimétrica.

In [12]:
artifact_summary = (tidy.groupby(["recording_id", "channel"], observed=True)
                    .agg(observaciones=("is_artifact", "size"), artefactos=("is_artifact", "sum"),
                         outliers_estadisticos=("is_stat_outlier", "sum")))
artifact_summary["% outliers"] = (100 * artifact_summary["outliers_estadisticos"] / artifact_summary["observaciones"]).round(1)
display(artifact_summary)
print("Motivos de artefacto:", tidy["artifact_reason"].value_counts().to_dict() or "ninguno")
print("Features más frecuentes entre los outliers estadísticos:")
display(tidy["outlier_features"].str.split(";").explode().value_counts().to_frame("filas"))

observaciones  artefactos  outliers_estadisticos  % outliers
recording_id channel                                                                 
SC4001       EEG Fpz-Cz            841           0                     44         5.2
             EEG Pz-Oz             841           0                     36         4.3
SC4002       EEG Fpz-Cz           1128           0                    107         9.5
             EEG Pz-Oz            1128           0                     76         6.7
SC4011       EEG Fpz-Cz           1103           0                     17         1.5
             EEG Pz-Oz            1103           0                     62         5.6
SC4012       EEG Fpz-Cz           1186           0                     20         1.7
             EEG Pz-Oz            1186           0                     65         5.5
SC4021       EEG Fpz-Cz           1025           0                     10         1.0
             EEG Pz-Oz            1025           0                     60         5.9
SC4022       EEG Fpz-Cz           1010           0                     31         3.1
             EEG Pz-Oz            1010           0                     41         4.1
SC4031       EEG Fpz-Cz            952           0                     31         3.3
             EEG Pz-Oz             952           0                    300        31.5
SC4032       EEG Fpz-Cz            911           0                    118        13.0
             EEG Pz-Oz             911           0                    143        15.7
SC4041       EEG Fpz-Cz           1236           0                    181        14.6
             EEG Pz-Oz            1236           0                     83         6.7
SC4042       EEG Fpz-Cz           1204           0                    193        16.0
             EEG Pz-Oz            1204           0                     96         8.0
SC4051       EEG Fpz-Cz            672           0                    154        22.9
             EEG Pz-Oz             672           0                    152        22.6
SC4052       EEG Fpz-Cz           1248           0                    180        14.4
             EEG Pz-Oz            1248           0                    190        15.2
SC4061       EEG Fpz-Cz            843           0                    172        20.4
             EEG Pz-Oz             843           0                    136        16.1
SC4062       EEG Fpz-Cz           1016           0                     75         7.4
             EEG Pz-Oz            1016           0                    236        23.2
SC4071       EEG Fpz-Cz            976           0                     92         9.4
             EEG Pz-Oz             976           0                     79         8.1
SC4072       EEG Fpz-Cz           1273           0                    290        22.8
             EEG Pz-Oz            1273           0                    226        17.8
SC4081       EEG Fpz-Cz           1134           0                     88         7.8
             EEG Pz-Oz            1134           0                    105         9.3
SC4082       EEG Fpz-Cz           1054           0                    325        30.8
             EEG Pz-Oz            1054           0                    101         9.6
SC4091       EEG Fpz-Cz           1143           0                     31         2.7
             EEG Pz-Oz            1143           0                     44         3.8
SC4092       EEG Fpz-Cz           1116           0                     91         8.2
             EEG Pz-Oz            1116           0                     88         7.9

Motivos de artefacto: ninguno
Features más frecuentes entre los outliers estadísticos:


,filas
outlier_features,
alpha_rel,2347
rms,1386
peak_to_peak,1274
total_power,443
delta_rel,442
delta_abs,441
spectral_entropy,392
sample_entropy,318


La figura siguiente responde: **¿dónde caen los valores extremos respecto del umbral físico de artefacto?** Los
puntos fuera de los bigotes son outliers estadísticos; la línea roja es el umbral de amplitud.

In [13]:
plots.amplitude_vs_threshold(obs_preview).show()

## Extracción de features

Para cada epoch de 30 s (3000 muestras a 100 Hz) y cada canal:

| Feature | Definición | Unidad | Implementación |
|---|---|---|---|
| PSD | Welch: ventana Hann de 4 s (400 muestras), 50 % de solapamiento, detrend constante. Resolución 0.25 Hz | µV²/Hz | `scipy.signal.welch` |
| Potencia absoluta de banda | $P_b = \sum_{f_1 \le f < f_2} \mathrm{PSD}(f)\,\Delta f$ | µV² | Propia (suma de la PSD) |
| Bandas | delta 0.5–4, theta 4–8, alfa 8–12, sigma 12–16, beta 16–30 Hz | Hz | `config.FREQ_BANDS` |
| Potencia total | $P_{tot} = \sum_{0.5 \le f < 30} \mathrm{PSD}(f)\,\Delta f$ | µV² | Propia |
| Potencia relativa | $P_b / P_{tot}$ | — | Propia |
| Spectral Entropy | $-\sum_f p_f \log_2 p_f / \log_2 N_f$, con $p_f$ = PSD de Welch normalizada | [0, 1] | `antropy.spectral_entropy` |
| Sample Entropy | $-\ln(A/B)$; $B$ = pares de subsecuencias de largo $m=2$ con distancia de Chebyshev ≤ $r = 0.2\,\mathrm{SD}$; $A$ = ídem con $m+1$ | — | `antropy.sample_entropy` |
| RMS | $\sqrt{\overline{x^2}}$ | µV | NumPy |
| Pico a pico | $\max x - \min x$ | µV | NumPy |

Decisiones:

- **Intervalos semiabiertos y regla del rectángulo**: las bandas son disjuntas, suman exactamente la potencia total y
  la potencia total es consistente con la varianza (Parseval). Integrar con Simpson (como hace YASA) da una delta
  absoluta ~10 % menor, porque pondera menos el extremo de 0.5 Hz, donde la PSD es máxima. Por eso los valores
  **absolutos** dependen de la convención de integración y no deben compararse con trabajos que usen otra; las
  **relativas** no tienen este problema.
- **Las entropías no se implementan a mano**: se usa `antropy`, una librería mantenida y validada.
- **Spectral Entropy** se calcula sobre la PSD completa (0–50 Hz) de la señal ya filtrada: `antropy` no admite
  restringir el rango, y por encima de 30 Hz la potencia filtrada es despreciable.
- **Potencia absoluta y relativa no son intercambiables**: la absoluta depende de la amplitud global (que varía entre
  sujetos y canales); la relativa describe la *forma* del espectro.
- **Sample Entropy y Spectral Entropy miden cosas distintas**: la primera mide la predecibilidad temporal de la
  forma de onda; la segunda, cuán repartida está la potencia en frecuencia.

In [14]:
feature_cols = features.FEATURE_COLUMNS
display(tidy[feature_cols].describe().T.round(3))

,count,mean,std,min,25%,50%,75%,max
delta_abs,42142.0,226.561,383.722,3.156,41.398,96.085,244.201,6276.006
theta_abs,42142.0,20.262,18.284,0.656,8.472,15.126,25.718,244.281
alpha_abs,42142.0,8.386,11.499,0.266,3.178,5.530,9.852,376.595
sigma_abs,42142.0,5.193,6.738,0.117,1.388,2.916,6.243,217.176
beta_abs,42142.0,4.638,12.735,0.117,0.956,1.759,3.693,954.916
delta_rel,42142.0,0.745,0.168,0.046,0.650,0.775,0.877,0.989
theta_rel,42142.0,0.122,0.069,0.004,0.068,0.112,0.165,0.652
alpha_rel,42142.0,0.066,0.088,0.001,0.020,0.041,0.075,0.864
sigma_rel,42142.0,0.033,0.034,0.000,0.011,0.022,0.041,0.405
beta_rel,42142.0,0.034,0.059,0.000,0.006,0.015,0.034,0.534


## Tidy Data

La tabla procesada cumple los principios de *tidy data*:

- cada **fila** es una observación: **un epoch de 30 s de un canal EEG**;
- cada **columna** es una variable: identificadores, contexto, features y flags;
- cada **tabla** es un tipo de unidad observacional. Los epochs completos del hipnograma y los registros van en
  tablas aparte (`epochs.parquet`, `recordings.parquet`).

Se guarda en **Parquet** (`data/processed/features_tidy.parquet`): es un formato columnar, comprimido, que conserva
los tipos (categorías, booleanos, float64) y se lee mucho más rápido que un CSV.

In [15]:
display(tidy.head())
print(tidy.dtypes.to_string())
print(f"\nMemoria: {tidy.memory_usage(deep=True).sum() / 1e6:.1f} MB · "
      f"Parquet: {pipeline.TIDY_PATH.stat().st_size / 1e6:.1f} MB")

,subject_id,recording_id,night,epoch_number,start_time_sec,sleep_stage,stage_original,channel,delta_abs,theta_abs,alpha_abs,sigma_abs,beta_abs,delta_rel,theta_rel,alpha_rel,sigma_rel,beta_rel,total_power,sample_entropy,spectral_entropy,rms,peak_to_peak,is_artifact,artifact_reason,is_stat_outlier,outlier_features
0,0,SC4001,1,961,28830.0,W,Sleep stage W,EEG Fpz-Cz,365.942101,35.133241,4.522651,3.764443,8.134511,0.876514,0.084152,0.010833,0.009017,0.019484,417.496948,0.841357,0.559640,20.605351,140.363800,False,None,False,None
1,0,SC4001,1,961,28830.0,W,Sleep stage W,EEG Pz-Oz,11.483911,3.862321,9.932861,0.781362,1.027928,0.423942,0.142582,0.366683,0.028845,0.037947,27.088383,1.074414,0.699609,5.310199,45.444906,False,None,False,None
2,0,SC4001,1,962,28860.0,W,Sleep stage W,EEG Fpz-Cz,538.662338,46.691065,4.449708,2.122842,8.673177,0.896875,0.077741,0.007409,0.003535,0.014441,600.599131,0.729980,0.555008,24.787359,175.951416,False,None,False,None
3,0,SC4001,1,962,28860.0,W,Sleep stage W,EEG Pz-Oz,22.670879,3.454855,3.684343,1.228974,1.580972,0.694999,0.105912,0.112947,0.037675,0.048466,32.620023,1.088807,0.629916,6.316598,59.920156,False,None,False,None
4,0,SC4001,1,963,28890.0,W,Sleep stage W,EEG Fpz-Cz,779.848960,67.036091,4.884337,2.252388,6.840181,0.905893,0.077871,0.005674,0.002616,0.007946,860.861957,0.553482,0.548740,30.808014,192.137085,False,None,False,None


subject_id             int64
recording_id          object
night                  int64
epoch_number           int64
start_time_sec       float64
sleep_stage         category
stage_original        object
channel             category
delta_abs            float64
theta_abs            float64
alpha_abs            float64
sigma_abs            float64
beta_abs             float64
delta_rel            float64
theta_rel            float64
alpha_rel            float64
sigma_rel            float64
beta_rel             float64
total_power          float64
sample_entropy       float64
spectral_entropy     float64
rms                  float64
peak_to_peak         float64
is_artifact             bool
artifact_reason       object
is_stat_outlier         bool
outlier_features      object

Memoria: 13.7 MB · Parquet: 6.4 MB


## SQLite OLTP

`data/database/sleep_oltp.sqlite` se reconstruye completo en cada ejecución: es un artefacto derivado de
`data/processed/`, y reconstruirlo garantiza que no queden filas obsoletas.

```
subject 1──N recording 1──N epoch N──1 hypnogram_label N──1 sleep_stage
                              │
                              1──N eeg_feature N──1 channel
```

Restricciones definidas:

- **PK** en todas las tablas; `eeg_feature` usa la clave compuesta (epoch_id, channel_id).
- **FK** activadas con `PRAGMA foreign_keys = ON`, que en SQLite están **desactivadas por defecto**.
- **UNIQUE** en (subject_id, night), (recording_id, epoch_number), nombres de canal, códigos de etapa y archivos.
- **NOT NULL** en todo identificador y atributo obligatorio. Las features admiten NULL a propósito, porque una
  feature no calculable es un faltante legítimo.
- **CHECK** de dominios: sexo ∈ {F, M}, potencias ≥ 0, Spectral Entropy ∈ [0, 1], y coherencia entre
  `is_artifact` y `artifact_reason`.
- **Índices** sobre las FK más consultadas.

In [16]:
database_oltp.build_oltp(tidy, recordings)
display(database_oltp.table_counts().to_frame())
show_checks(validation.validate_oltp(tidy))

INFO src.database_oltp: OLTP creada: data/database/sleep_oltp.sqlite


,rows
subject,10
recording,20
channel,2
sleep_stage,7
hypnogram_label,8
epoch,21071
eeg_feature,42142


  ✔ PRAGMA foreign_keys = ON
  ✔ PRAGMA foreign_key_check sin violaciones
  ✔ recording: 20 filas
  ✔ epoch: 21071 filas = epochs de la tidy
  ✔ eeg_feature: 42142 filas = filas de la tidy
  ✔ Potencia relativa recalculada en la vista coincide (max |dif| = 1.9e-15)


In [17]:
# Demostración: las restricciones se aplican (cada intento se revierte).
import sqlite3
with database_oltp.connect() as conn:
    for description, sql in [
        ("epoch de un registro inexistente", "INSERT INTO epoch (recording_id, epoch_number, label_id) VALUES ('SC9999', 0, 1)"),
        ("(epoch, canal) duplicado", "INSERT INTO eeg_feature (epoch_id, channel_id, is_artifact) VALUES (1, 1, 0)"),
        ("artefacto sin motivo", "INSERT INTO eeg_feature (epoch_id, channel_id, is_artifact) VALUES (1, 2, 1)"),
    ]:
        try:
            conn.execute(sql)
            print(f"  ✘ {description}: NO fue rechazado")
        except sqlite3.IntegrityError as error:
            print(f"  ✔ {description}: rechazado ({error})")
    conn.rollback()
conn.close()

  ✔ epoch de un registro inexistente: rechazado (FOREIGN KEY constraint failed)
  ✔ (epoch, canal) duplicado: rechazado (UNIQUE constraint failed: eeg_feature.epoch_id, eeg_feature.channel_id)
  ✔ artefacto sin motivo: rechazado (CHECK constraint failed: (is_artifact = 0 AND artifact_reason IS NULL) OR (is_artifact = 1 AND artifact_reason IS NOT NULL))


## Normalización

**1NF.** Todos los atributos son atómicos (un valor por celda; no hay listas ni grupos repetidos) y cada tabla tiene
clave primaria. Las bandas de frecuencia se modelaron como columnas fijas (`delta_abs`, …) y no como un grupo
repetido: son un conjunto cerrado y conocido de variables de la misma observación.

**2NF.** No hay dependencias parciales de una clave compuesta:

- en `eeg_feature` (PK = epoch_id + channel_id), cada feature depende del par completo: la potencia delta del epoch
  100 en Fpz-Cz no está determinada solo por el epoch ni solo por el canal;
- `start_sec = epoch_number × 30` dependería solo de `epoch_number`, que es parte de la clave candidata
  (recording_id, epoch_number). Por eso **no se almacena** y se calcula en la vista `v_epoch`;
- el flag de artefacto depende del canal (un parpadeo afecta a Fpz-Cz y no a Pz-Oz), por eso vive en `eeg_feature`
  y no en `epoch`.

**3NF.** No hay dependencias transitivas entre atributos no clave:

- la etiqueta original del hipnograma determina la etapa ("Sleep stage 4" → N3). Guardar ambas en `epoch` crearía
  `epoch → etiqueta → etapa`; por eso existe `hypnogram_label`, y `epoch` solo referencia la etiqueta;
- edad y sexo dependen del sujeto, no del registro: están en `subject`;
- la potencia total y las relativas se derivan de las absolutas (total = Σ bandas; relativa = banda / total).
  Almacenarlas sería redundante; las calcula la vista `v_eeg_feature`. La validación anterior confirma que la vista
  reproduce los valores de la tabla tidy (diferencia ~1e-15);
- los outliers estadísticos dependen de la distribución de todo el conjunto, no de la fila: no son atributos de la
  observación y se calculan en el ETL hacia el Data Warehouse.

## Data Warehouse

```
SQLite OLTP (3NF)
    ↓  Extract: una consulta SQL con JOIN sobre las vistas del OLTP
    ↓  Transform: dimensiones con surrogate keys, outliers estadísticos, tiempo desde el inicio del sueño
    ↓  Load: dimensiones primero, luego hechos (orden requerido por las FK)
SQLite Data Warehouse (estrella)
```

El ETL lee **del OLTP**, no de los Parquet: el OLTP es la fuente de verdad transaccional y el DW, su modelo analítico
derivado.

In [18]:
dw_counts = database_dw.build_dw()
display(pd.Series(dw_counts, name="filas").to_frame())
show_checks(validation.validate_dw())

INFO src.database_dw: DW creado: data/database/sleep_dw.sqlite


,filas
dim_subject,10
dim_recording,20
dim_channel,2
dim_sleep_stage,8
fact_eeg_epoch,42142


  ✔ DW: PRAGMA foreign_key_check sin violaciones
  ✔ fact_eeg_epoch: 42142 filas = eeg_feature del OLTP
  ✔ Grano respetado: (registro, epoch, canal) único en la tabla de hechos
  ✔ Cada registro pertenece a un único sujeto
  ✔ 8 consultas analíticas reproducibles


## Esquema estrella

**Grano de `fact_eeg_epoch`:** *una observación de features EEG correspondiente a un epoch de 30 segundos, un canal,
un sujeto y un registro.*

```
                    dim_subject
                         │
   dim_recording ── fact_eeg_epoch ── dim_channel
                         │
                  dim_sleep_stage
```

| Tabla | Contenido |
|---|---|
| `fact_eeg_epoch` | Claves de las 4 dimensiones, `epoch_number` (dimensión degenerada), `start_sec`, `hours_since_sleep_onset`, las 15 métricas y los flags de calidad |
| `dim_subject` | Edad, sexo, grupo etario |
| `dim_recording` | Noche, inicio, apagado de luces, duración total y analizada |
| `dim_channel` | Nombre, nombre corto, ubicación |
| `dim_sleep_stage` | Grano = etiqueta original del hipnograma, con su código AASM (S3 y S4 → N3): la jerarquía queda desnormalizada en la dimensión |

Decisiones:

- **Surrogate keys** enteras y asignadas en un orden determinístico, para que el DW sea idéntico en cada ejecución.
- **Las dimensiones no se referencian entre sí** (eso sería un copo de nieve): la tabla de hechos tiene las 4 claves.
- **Redundancia deliberada**: a diferencia del OLTP, se materializan las potencias total y relativas, el tiempo desde
  el inicio del sueño y los flags de outlier. Así las consultas analíticas necesitan un solo JOIN por dimensión.
- `hours_since_sleep_onset` permite comparar noches que empiezan a horas distintas.

In [19]:
with database_dw.connect() as conn:
    for table in ["dim_subject", "dim_recording", "dim_channel", "dim_sleep_stage"]:
        display(Markdown(f"**{table}**"))
        display(pd.read_sql(f"SELECT * FROM {table}", conn))
conn.close()

**dim_subject**

,subject_key,subject_id,age,sex,age_group
0,1,0,33,F,<40
1,2,1,33,F,<40
2,3,2,26,F,<40
3,4,3,26,F,<40
4,5,4,34,F,<40
5,6,5,28,F,<40
6,7,6,31,F,<40
7,8,7,30,F,<40
8,9,8,25,F,<40
9,10,9,25,F,<40


**dim_recording**

,recording_key,recording_id,night,start_datetime,lights_off,sampling_frequency_hz,psg_duration_h,analyzed_duration_h
0,1,SC4001,1,1989-04-24T16:13:00,00:38:00,100.0,22.083333,7.008333
1,2,SC4002,2,1989-04-25T14:50:00,21:57:00,100.0,23.583333,9.400000
2,3,SC4011,1,1989-03-29T16:49:00,22:44:00,100.0,23.350000,9.191667
3,4,SC4012,2,1989-03-30T17:00:00,22:15:00,100.0,23.750000,9.883333
4,5,SC4021,1,1989-04-05T16:48:00,22:50:00,100.0,23.366667,8.541667
5,6,SC4022,2,1989-04-06T16:42:00,22:57:00,100.0,22.966667,8.416667
6,7,SC4031,1,1989-04-10T16:50:00,00:02:00,100.0,23.500000,7.933333
7,8,SC4032,2,1989-04-11T16:49:00,00:24:00,100.0,22.766667,7.591667
8,9,SC4041,1,1989-04-24T16:50:00,23:12:00,100.0,21.416667,10.300000
9,10,SC4042,2,1989-04-25T15:04:00,23:35:00,100.0,23.266667,10.033333


**dim_channel**

,channel_key,channel_name,short_name,location
0,1,EEG Fpz-Cz,Fpz-Cz,Bipolar frontal-central (Fpz referenced to Cz)
1,2,EEG Pz-Oz,Pz-Oz,Bipolar parietal-occipital (Pz referenced to Oz)


**dim_sleep_stage**

,stage_key,original_description,stage_code,stage_name,is_scored,stage_order
0,1,Sleep stage W,W,Wake,1,1
1,2,Sleep stage 1,N1,NREM 1,1,2
2,3,Sleep stage 2,N2,NREM 2,1,3
3,4,Sleep stage 3,N3,NREM 3 (slow-wave sleep),1,4
4,5,Sleep stage 4,N3,NREM 3 (slow-wave sleep),1,4
5,6,Sleep stage R,REM,Rapid eye movement,1,5
6,7,Movement time,MOVE,Movement time (unscored),0,6
7,8,Sleep stage ?,UNK,Unknown (unscored),0,7


## Consultas SQL

Las consultas están en `src/queries.py`. Comparten dos convenciones:

- `:include_artifacts` (0/1) controla si se incluyen las observaciones con artefacto;
- las comparaciones entre etapas usan solo etapas puntuadas (`is_scored = 1`). Las consultas de calidad (conteos y
  porcentaje de artefactos) describen todo el conjunto.

Como ejemplo, la consulta 1 completa:

In [20]:
print(queries.QUERIES["sample_entropy_by_stage"])


        SELECT st.stage_code, c.short_name AS channel,
               COUNT(f.sample_entropy) AS n, AVG(f.sample_entropy) AS mean_sample_entropy,
               MIN(f.sample_entropy) AS min_value, MAX(f.sample_entropy) AS max_value
        
FROM fact_eeg_epoch f
JOIN dim_subject     su ON su.subject_key   = f.subject_key
JOIN dim_recording   r  ON r.recording_key  = f.recording_key
JOIN dim_channel     c  ON c.channel_key    = f.channel_key
JOIN dim_sleep_stage st ON st.stage_key     = f.stage_key
 WHERE st.is_scored = 1 AND (:include_artifacts = 1 OR f.is_artifact = 0)
        GROUP BY st.stage_code, st.stage_order, c.short_name
        ORDER BY st.stage_order, channel


In [21]:
# 1. Sample Entropy promedio por etapa
q1 = queries.run_query("sample_entropy_by_stage")
display(q1.round(3))

,stage_code,channel,n,mean_sample_entropy,min_value,max_value
0,W,Fpz-Cz,3656,0.842,0.169,1.910
1,W,Pz-Oz,3656,1.297,0.117,2.019
2,N1,Fpz-Cz,1689,1.039,0.326,1.925
3,N1,Pz-Oz,1689,0.962,0.154,1.697
4,N2,Fpz-Cz,8927,0.762,0.309,1.533
5,N2,Pz-Oz,8927,0.781,0.224,1.428
6,N3,Fpz-Cz,3074,0.508,0.306,0.909
7,N3,Pz-Oz,3074,0.537,0.134,0.943
8,REM,Fpz-Cz,3693,0.958,0.307,1.643
9,REM,Pz-Oz,3693,0.947,0.106,1.394


In [22]:
# 2. Spectral Entropy promedio por etapa
q2 = queries.run_query("spectral_entropy_by_stage")
display(q2.round(3))

,stage_code,channel,n,mean_spectral_entropy,min_value,max_value
0,W,Fpz-Cz,3656,0.557,0.258,0.877
1,W,Pz-Oz,3656,0.703,0.242,0.921
2,N1,Fpz-Cz,1689,0.618,0.306,0.867
3,N1,Pz-Oz,1689,0.643,0.264,0.836
4,N2,Fpz-Cz,8927,0.598,0.315,0.819
5,N2,Pz-Oz,8927,0.616,0.290,0.826
6,N3,Fpz-Cz,3074,0.483,0.355,0.621
7,N3,Pz-Oz,3074,0.496,0.313,0.688
8,REM,Fpz-Cz,3693,0.617,0.276,0.840
9,REM,Pz-Oz,3693,0.651,0.264,0.800


In [23]:
# 3. Potencia relativa de cada banda por etapa
q3 = queries.run_query("relative_band_power_by_stage")
display(q3.round(3))

,stage_code,channel,n,delta,theta,alpha,sigma,beta
0,W,Fpz-Cz,3656,0.805,0.081,0.053,0.015,0.046
1,W,Pz-Oz,3656,0.475,0.114,0.201,0.055,0.155
2,N1,Fpz-Cz,1689,0.713,0.133,0.070,0.024,0.060
3,N1,Pz-Oz,1689,0.626,0.158,0.145,0.028,0.044
4,N2,Fpz-Cz,8927,0.788,0.112,0.038,0.047,0.015
5,N2,Pz-Oz,8927,0.739,0.150,0.051,0.046,0.014
6,N3,Fpz-Cz,3074,0.931,0.046,0.012,0.009,0.003
7,N3,Pz-Oz,3074,0.902,0.068,0.018,0.009,0.003
8,REM,Fpz-Cz,3693,0.732,0.158,0.055,0.019,0.037
9,REM,Pz-Oz,3693,0.657,0.182,0.108,0.023,0.029


In [24]:
# 4. Comparación entre canales
q4 = queries.run_query("channel_comparison")
display(q4.round(3))

,channel,stage_code,n,sample_entropy,spectral_entropy,delta_rel,alpha_rel,total_power,rms
0,Fpz-Cz,W,3656,0.842,0.557,0.805,0.053,513.711,21.810
1,Pz-Oz,W,3656,1.297,0.703,0.475,0.201,129.108,10.890
2,Fpz-Cz,N1,1689,1.039,0.618,0.713,0.070,113.918,10.562
3,Pz-Oz,N1,1689,0.962,0.643,0.626,0.145,83.095,8.597
4,Fpz-Cz,N2,8927,0.762,0.598,0.788,0.038,236.274,15.229
5,Pz-Oz,N2,8927,0.781,0.616,0.739,0.051,134.200,11.287
6,Fpz-Cz,N3,3074,0.508,0.483,0.931,0.012,1059.591,31.989
7,Pz-Oz,N3,3074,0.537,0.496,0.902,0.018,412.596,20.175
8,Fpz-Cz,REM,3693,0.958,0.617,0.732,0.055,107.463,10.480
9,Pz-Oz,REM,3693,0.947,0.651,0.657,0.108,60.228,7.683


In [25]:
# 5. Comparación entre sujetos
q5 = queries.run_query("subject_comparison")
display(q5.round(3).head(20))

,subject_id,age,sex,stage_code,channel,n,sample_entropy,spectral_entropy,delta_rel
0,0,33,F,W,Fpz-Cz,371,0.925,0.596,0.715
1,0,33,F,W,Pz-Oz,371,1.072,0.650,0.493
2,0,33,F,N1,Fpz-Cz,117,0.980,0.658,0.665
3,0,33,F,N1,Pz-Oz,117,0.858,0.637,0.627
4,0,33,F,N2,Fpz-Cz,623,0.722,0.598,0.815
5,0,33,F,N2,Pz-Oz,623,0.709,0.596,0.782
6,0,33,F,N3,Fpz-Cz,517,0.504,0.492,0.928
7,0,33,F,N3,Pz-Oz,517,0.513,0.495,0.907
8,0,33,F,REM,Fpz-Cz,340,0.987,0.658,0.666
9,0,33,F,REM,Pz-Oz,340,0.882,0.638,0.607


In [26]:
# 6. Cantidad de epochs por etapa
q6 = queries.run_query("epochs_by_stage")
display(q6.round(3))

,stage_code,n_epochs,percent,hours
0,W,3656,17.35,30.47
1,N1,1689,8.02,14.07
2,N2,8927,42.37,74.39
3,N3,3074,14.59,25.62
4,REM,3693,17.53,30.77
5,MOVE,32,0.15,0.27


In [27]:
# 7. Porcentaje de epochs marcados como artefactos
q7 = queries.run_query("artifact_percentage")
display(q7.round(3))

,recording_id,channel,n_observations,n_artifacts,artifact_percent,n_stat_outliers,stat_outlier_percent
0,SC4001,Fpz-Cz,841,0,0.0,44,5.23
1,SC4001,Pz-Oz,841,0,0.0,36,4.28
2,SC4002,Fpz-Cz,1128,0,0.0,107,9.49
3,SC4002,Pz-Oz,1128,0,0.0,76,6.74
4,SC4011,Fpz-Cz,1103,0,0.0,17,1.54
5,SC4011,Pz-Oz,1103,0,0.0,62,5.62
6,SC4012,Fpz-Cz,1186,0,0.0,20,1.69
7,SC4012,Pz-Oz,1186,0,0.0,65,5.48
8,SC4021,Fpz-Cz,1025,0,0.0,10,0.98
9,SC4021,Pz-Oz,1025,0,0.0,60,5.85


In [28]:
# 8. Evolución temporal durante la noche
q8 = queries.run_query("temporal_evolution")
display(q8.round(3).head(20))

,recording_id,channel,hour_bin,n,sample_entropy,spectral_entropy,delta_rel,pct_n3,pct_rem
0,SC4001,Fpz-Cz,-1,60,0.755,0.555,0.810,0.0,0.0
1,SC4001,Fpz-Cz,0,120,0.561,0.537,0.888,67.5,0.0
2,SC4001,Fpz-Cz,1,120,0.651,0.561,0.819,42.5,24.2
3,SC4001,Fpz-Cz,2,120,0.620,0.542,0.848,34.2,24.2
4,SC4001,Fpz-Cz,3,120,0.640,0.563,0.839,28.3,7.5
5,SC4001,Fpz-Cz,4,120,0.771,0.606,0.759,6.7,13.3
6,SC4001,Fpz-Cz,5,120,0.804,0.593,0.749,4.2,35.0
7,SC4001,Fpz-Cz,6,61,0.690,0.527,0.856,0.0,0.0
8,SC4001,Pz-Oz,-1,60,1.215,0.665,0.567,0.0,0.0
9,SC4001,Pz-Oz,0,120,0.575,0.538,0.857,67.5,0.0


In [29]:
# Reproducibilidad: todas las consultas devuelven exactamente lo mismo al repetirse.
for name in queries.QUERIES:
    pd.testing.assert_frame_equal(queries.run_query(name), queries.run_query(name))
print(f"✔ {len(queries.QUERIES)} consultas reproducibles")

✔ 8 consultas reproducibles


## EDA

Todas las figuras se construyen con datos del **Data Warehouse** (vía `src/queries.py`), no desde los EDF. Cada una
responde una pregunta concreta.

**Paleta**: W naranja; N1→N3 una rampa azul de claro a oscuro (profundidad creciente del sueño); REM aqua. Se validó
para visión normal y daltonismo (protanopía/deuteranopía) en todos los pares de colores.

In [30]:
obs = queries.load_observations()
print(f"Observaciones cargadas desde el DW: {len(obs):,}")

Observaciones cargadas desde el DW: 42,142


**¿Cuánto tiempo pasa en cada etapa dentro de la ventana de análisis?**

In [31]:
plots.stage_distribution(queries.run_query("epochs_by_stage")).show()

**¿Cómo cambia la Sample Entropy entre etapas, y es igual en ambos canales?**

In [32]:
plots.metric_by_stage(obs[~obs["is_artifact"]], "sample_entropy").show()

**¿Cómo cambia la Spectral Entropy entre etapas?**

In [33]:
plots.metric_by_stage(obs[~obs["is_artifact"]], "spectral_entropy").show()

**¿Qué bandas dominan cada etapa? (canal posterior, donde el alfa de vigilia es más visible)**

In [34]:
rel = queries.run_query("relative_band_power_by_stage")
plots.band_power_by_stage(rel, "Pz-Oz").show()

**¿En qué etapa es cada banda relativamente más fuerte? El color es el z-score dentro de cada banda; el texto, la potencia relativa media.**

In [35]:
for ch in ["Fpz-Cz", "Pz-Oz"]:
    plots.stage_band_heatmap(rel, ch).show()

**¿Se asocian la actividad delta y la complejidad? ¿La relación depende de la etapa?**

In [36]:
clean = obs[~obs["is_artifact"]]
for ch in ["Fpz-Cz", "Pz-Oz"]:
    plots.delta_vs_entropy(clean, ch, max_points_per_stage=1500).show()

**¿Difieren el canal frontal y el posterior?**

In [37]:
plots.channel_comparison(queries.run_query("channel_comparison"), "sample_entropy").show()
plots.channel_comparison(queries.run_query("channel_comparison"), "alpha_rel").show()

**¿El patrón entre etapas se repite en cada sujeto, o lo produce un sujeto en particular?**

In [38]:
plots.subject_comparison(queries.run_query("subject_comparison"), "sample_entropy", "Pz-Oz").show()

**¿Cómo evolucionan la delta relativa y la complejidad a lo largo de la noche?**

In [39]:
evo = queries.run_query("temporal_evolution")
plots.temporal_evolution(evo, "delta_rel", "Fpz-Cz").show()
plots.temporal_evolution(evo, "sample_entropy", "Fpz-Cz").show()

**¿Cuán fuerte es la relación lineal entre las métricas, y cuánto de ella se explica por la etapa?** Se compara la
correlación de Spearman global con la mediana de las correlaciones dentro de cada etapa: si la correlación global es
mucho mayor, gran parte de la asociación se debe a que ambas variables cambian con la etapa (confusión), no a una
relación dentro de la etapa.

In [40]:
pairs = [("delta_rel", "sample_entropy"), ("delta_rel", "spectral_entropy"), ("sample_entropy", "spectral_entropy")]
scored = obs[obs["is_scored"] & ~obs["is_artifact"]]
rows = []
for ch, g in scored.groupby("channel"):
    for a, b in pairs:
        within = g.groupby("stage_code").apply(lambda s: s[a].corr(s[b], method="spearman"), include_groups=False)
        rows.append({"canal": ch, "par": f"{a} ~ {b}", "ρ global": g[a].corr(g[b], method="spearman"),
                     "ρ dentro de etapa (mediana)": within.median()})
display(pd.DataFrame(rows).round(2))

,canal,par,ρ global,ρ dentro de etapa (mediana)
0,Fpz-Cz,delta_rel ~ sample_entropy,-0.88,-0.87
1,Fpz-Cz,delta_rel ~ spectral_entropy,-0.94,-0.93
2,Fpz-Cz,sample_entropy ~ spectral_entropy,0.82,0.81
3,Pz-Oz,delta_rel ~ sample_entropy,-0.90,-0.75
4,Pz-Oz,delta_rel ~ spectral_entropy,-0.92,-0.89
5,Pz-Oz,sample_entropy ~ spectral_entropy,0.90,0.88


## Dashboard

`src/app.py` es un dashboard Streamlit que lee **solo** `data/database/sleep_dw.sqlite`.

- **Filtros**: sujeto, noche/registro, canal, etapas de sueño, incluir/excluir artefactos.
- **Contenido**: hipnograma; Sample Entropy y Spectral Entropy por etapa; potencia relativa por banda; heatmap
  etapa × banda; scatter delta vs Sample Entropy coloreado por etapa.

Ejecución local, desde la raíz del repositorio:

```bash
streamlit run src/app.py
```

En Colab, Streamlit necesita un túnel para exponer el puerto; las mismas figuras ya se muestran en la sección EDA de
este notebook. Ver el README para la opción con túnel.

## Resultados principales

Muestra: 10 sujetos × 2 noches = 20 registros, 21 071 epochs analizados (175 h) y 42 142 observaciones
epoch × canal. Los valores citados corresponden a esta ejecución y se recalculan en las celdas anteriores.
**Todos los resultados son descriptivos.**

**1. Distribución de etapas.** Dentro de la ventana de análisis, N2 es la etapa más frecuente (≈42 % de los epochs),
seguida de REM, W y N3 (≈15–18 % cada una) y N1 (≈8 %). Hubo 32 epochs de *Movement time*; no se observó ningún
`?` dentro de la ventana.

**2. Complejidad temporal (Sample Entropy).** Baja de forma monótona con la profundidad del sueño NREM, N1 > N2 > N3
(en Pz-Oz, ≈0.96 → 0.78 → 0.54), y REM queda en niveles similares a N1 (≈0.95). El orden N1 > N2 > N3 y REM > N2
se cumple en **los 10 sujetos y en ambos canales**, así que el patrón no depende de un sujeto en particular. La
vigilia muestra la mayor complejidad en Pz-Oz (≈1.30), pero en Fpz-Cz es menor y muy variable entre sujetos
(0.46–1.08). Esto es compatible con la contaminación por movimientos oculares, que agrega actividad lenta y regular en
el canal frontal.

**3. Dispersión espectral (Spectral Entropy).** N3 tiene la menor Spectral Entropy (≈0.48–0.50): la potencia se
concentra en pocas frecuencias. Ocurre en 19 de las 20 combinaciones sujeto × canal; la excepción es Fpz-Cz del
sujeto 1, donde W es aún menor (otra vez, actividad ocular). Las dos entropías cuentan historias parecidas pero no
idénticas: miden propiedades distintas de la señal.

**4. Actividad espectral.**

- La **delta relativa** crece con la profundidad del sueño: ≈0.90–0.93 en N3.
- El **alfa relativo** es máximo en vigilia en el canal posterior (Pz-Oz ≈0.20, contra ≈0.05 en Fpz-Cz), coherente
  con el ritmo alfa occipital de la vigilia relajada.
- **Theta** es relativamente más alta en REM y N1.
- **Sigma** destaca en N2, la banda de los husos de sueño.
- **Beta** es máxima en vigilia.

**5. Delta y complejidad.** La delta relativa y la Sample Entropy tienen una fuerte asociación negativa (Spearman
ρ ≈ −0.9). La correlación se mantiene **dentro de cada etapa** (ρ ≈ −0.7 a −0.94, salvo W en Pz-Oz), así que no se
explica solo por el cambio de etapa: dentro de un mismo estado, los epochs con más potencia lenta son más
predecibles. Es una asociación entre dos descriptores de la misma señal, **no una relación causal**.

**6. Evolución nocturna.** El porcentaje de N3 es máximo en las primeras horas tras el inicio del sueño (≈30 % en la
primera hora) y cae a <10 % después de la hora 4, mientras REM aumenta hacia la segunda mitad de la noche. En
paralelo, la Sample Entropy media es más baja en las horas 1–3 y aumenta después. Este es el patrón clásico de la
arquitectura del sueño, con más sueño de ondas lentas al principio y más REM al final.

**7. Calidad de datos.**

- Ninguna observación superó los criterios físicos de artefacto: el pico a pico máximo fue de ≈435 µV, contra el
  umbral de 500 µV. Sleep-EDF es un dataset curado; los artefactos más sutiles (oculares, musculares) no se detectan
  con criterios de amplitud. Es una limitación, no una prueba de ausencia de artefactos.
- En cambio, ≈11 % de las observaciones son outliers estadísticos según la regla de Tukey. En algunos registros
  llegan al 23–31 %, porque los cuartiles se calculan sobre todos los registros juntos y las diferencias de amplitud
  entre sujetos y noches aparecen como "outliers". Es la ilustración directa de que **un outlier estadístico no es
  un artefacto**: eliminarlos habría descartado registros completos de fisiología normal.

**En síntesis:** al profundizarse el sueño NREM, el EEG se vuelve más lento (más delta), más regular (menor Sample
Entropy) y con la potencia más concentrada (menor Spectral Entropy). REM y N1 se parecen a la vigilia en complejidad,
pero se distinguen de ella por el alfa posterior. Estas observaciones coinciden con lo conocido sobre la fisiología
del sueño, lo que funciona como validación del pipeline completo.

## Limitaciones

- **Muestra pequeña y homogénea**: 10 sujetos sanos de un único estudio. Con la regla de selección por defecto
  (los primeros N sujetos por ID), los 10 resultan **mujeres de 25 a 34 años**, aunque el dataset completo está
  balanceado por sexo y abarca de 25 a 101 años. Los resultados no se generalizan a otras edades, al sexo masculino
  ni a patologías. Para una muestra más diversa basta con definir `SUBJECT_IDS` en `config.py`.
- **Sin inferencia estadística**: los epochs de una misma noche no son independientes (autocorrelación temporal y
  anidamiento sujeto/noche). Un análisis inferencial necesitaría modelos mixtos con sujeto como efecto aleatorio.
- **Puntuación R&K**: los hipnogramas siguen Rechtschaffen & Kales (1968); unir S3+S4 en N3 aproxima AASM, pero
  los criterios no son idénticos.
- **Dos canales bipolares**: Fpz-Cz y Pz-Oz no permiten análisis topográfico. Fpz-Cz es sensible a la actividad
  ocular, que cae en la banda delta, sobre todo en vigilia.
- **Detección de artefactos conservadora**: los criterios físicos (amplitud, señal plana, no finitos) no detectan
  artefactos sutiles, como movimientos oculares o EMG de baja amplitud. No se aplicaron métodos avanzados (ICA,
  regresión de EOG) para evitar sobreprocesar la señal.
- **Convenciones de las métricas**: la potencia absoluta depende del método de integración, y la Sample Entropy
  depende de m, r y la longitud del epoch. Las comparaciones son válidas dentro de este pipeline, no necesariamente
  con valores publicados que usen otros parámetros.
- **Recorte de vigilia**: conservar ±30 min es una convención. Otra ventana cambiaría la proporción de W y sus
  estadísticos.

## Reproducibilidad

| Aspecto | Cómo se garantiza |
|---|---|
| Datos | Descarga programática desde la fuente pública; verificación SHA256 contra los hashes oficiales; manifest |
| Rutas | `pathlib`, relativas a la raíz del repositorio; sin rutas absolutas ni archivos privados |
| Parámetros | Centralizados en `src/config.py`; su hash se guarda con cada resultado intermedio |
| Caché | Por registro, invalidado automáticamente si cambian parámetros o versión del código |
| Aleatoriedad | La única aleatoriedad (submuestreo para el scatter) usa `RANDOM_SEED` |
| Bases | Se reconstruyen desde cero en cada ejecución; surrogate keys determinísticas |
| Consultas | Verificadas como reproducibles (mismo resultado en ejecuciones repetidas) |
| Entorno | Versiones registradas en `data/processed/environment.json` |

In [41]:
env = json.loads(pipeline.ENVIRONMENT_PATH.read_text())
display(pd.Series(env["library_versions"], name="versión").to_frame())
print("Hash de parámetros del Transform:", env["params_hash"])

,versión
python,3.13.11
numpy,2.2.6
pandas,2.2.3
scipy,1.15.3
mne,1.9.0
mne-bids,0.20.0
antropy,0.1.9
pyarrow,25.0.1
plotly,7.1.0
streamlit,1.65.0


Hash de parámetros del Transform: 63085a8cb58e


## Conclusiones

Se construyó un pipeline de Ingeniería de Datos reproducible de punta a punta:

- **Extract** desde una fuente pública, con integridad verificada.
- **Organización BIDS** que separa los datos crudos de su representación estandarizada.
- **Transform** documentado: epochs, recorte, filtro y features con librerías científicas mantenidas.
- **Control de calidad** explícito, que distingue datos faltantes, outliers estadísticos y artefactos de señal sin
  eliminar ni imputar información.
- **Modelado relacional 3NF**, con restricciones verificadas.
- **Data Warehouse en estrella**, alimentado por un ETL desde el OLTP.
- **Consultas SQL reproducibles**, visualizaciones y un dashboard que leen únicamente del DW.

Sobre la pregunta del caso de estudio, los datos muestran de forma consistente entre sujetos y canales que, a medida
que el sueño NREM se profundiza, la potencia se concentra en delta y la complejidad (en sus dos acepciones) disminuye.
REM y N1 se parecen más a la vigilia en complejidad, pero con menos alfa posterior que la vigilia relajada. Estas
observaciones son descriptivas y coinciden con lo esperado por la fisiología del sueño, lo que funciona también como
validación del pipeline.

### Trabajo futuro: almacenamiento de EEG continuo

Este trabajo almacena **features por epoch**: un volumen pequeño que SQLite maneja sin dificultad. Si el objetivo
fuera almacenar **señal EEG continua**, por ejemplo con adquisición en tiempo real o *wearables* de sueño que
transmiten muestras de forma permanente, convendría una base de series temporales como **TimescaleDB** (extensión de
PostgreSQL), que ofrece:

- particionado automático por tiempo (*hypertables*);
- compresión columnar;
- agregados continuos (por ejemplo, potencia por banda cada 30 s calculada en la base);
- políticas de retención.

No es necesaria para este TP, cuyo foco es el modelado OLTP/DW de métricas derivadas.